In [ ]:
from google.colab import drive

drive.mount("/content/drive")


# Coleta de apartes do Plenario do Senado

Notebook Colab para clonar o repositorio, instalar dependencias e executar o coletor metadata-only de apartes do Plenario do Senado salvando dados no Google Drive.

A primeira celula monta o Drive antes de qualquer codigo do projeto. Os dados completos ficam em `MyDrive/falando_nela/data`; o clone do repositorio fica em `/content/falando_nela` e pode ser recriado a cada sessao.

A coleta grava respostas oficiais em `raw/senado/plenario_apartes/metadata/` e nao cria corpus textual em `ano=YYYY/mes=MM/`.


In [ ]:
import os
from pathlib import Path

DATA_ROOT = Path("/content/drive/MyDrive/falando_nela/data")
os.environ["FALANDO_NELA_DATA_ROOT"] = str(DATA_ROOT)

for name in ["raw", "checkpoints", "logs", "manifests", "processed"]:
    (DATA_ROOT / name).mkdir(parents=True, exist_ok=True)

print("FALANDO_NELA_DATA_ROOT=", os.environ["FALANDO_NELA_DATA_ROOT"])


In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/pedblan/falando_nela.git"
REPO_DIR = Path("/content/falando_nela")
REPO_REF = ""  # opcional: branch, tag ou commit. Deixe vazio para usar o default remoto.

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all", "--tags", "--prune"], check=True)
    if not REPO_REF:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

if REPO_REF:
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_REF], check=True)

subprocess.run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL], check=True)
os.chdir(REPO_DIR)
print("Repo em:", Path.cwd())
subprocess.run(["git", "status", "--short"], check=True)


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)


## Parametros

Use a validacao curta primeiro. Ela roda em modo producao para confirmar escrita no Drive, mas passa `--sample` e `--sample-limit` para limitar a execucao. A coleta completa fica protegida por uma flag.


In [ ]:
from datetime import datetime, timezone

DATA_INICIO_VALIDACAO = "2025-01-01"
DATA_FIM_VALIDACAO = "2025-01-31"
SAMPLE_LIMIT_VALIDACAO = 10

# Janela ampla para maximizar cobertura historica; recorte analitico recomendado: 2010-01-01 em diante.
DATA_INICIO_PROD = "1900-01-01"
DATA_FIM_PROD = "2026-05-18"

RUN_STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_ID_VALIDACAO = f"colab-senado-plenario-apartes-validacao-{RUN_STAMP}"
RUN_ID_PROD = "prod-senado-plenario-apartes-baseline"

print("RUN_ID_VALIDACAO=", RUN_ID_VALIDACAO)
print("RUN_ID_PROD=", RUN_ID_PROD, "# fixo para permitir retomada com --resume")


In [ ]:
MODULE = "coleta.senado.plenario_apartes.collect"
SOURCE = "senado"
DATASET = "plenario_apartes"
RECORD_TYPE = "senador_apartes_metadata"


def summarize_payload(payload):
    parlamentar = payload.get("ApartesParlamentar", {}).get("Parlamentar", {})
    apartes = parlamentar.get("Apartes") if isinstance(parlamentar, dict) else None
    if isinstance(apartes, dict):
        aparte_obj = apartes.get("Aparte")
        if isinstance(aparte_obj, list):
            total_apartes = len(aparte_obj)
        elif isinstance(aparte_obj, dict):
            total_apartes = 1
        else:
            total_apartes = 0
    else:
        total_apartes = 0
    identificacao = parlamentar.get("IdentificacaoParlamentar", {}) if isinstance(parlamentar, dict) else {}
    return {
        "parlamentar": identificacao.get("NomeParlamentar"),
        "codigo_parlamentar": identificacao.get("CodigoParlamentar"),
        "total_apartes_payload": total_apartes,
        "apartes_null": apartes is None,
    }

import json
import subprocess
import sys
from pathlib import Path


def run_collector(run_id, data_inicio, data_fim, sample=False, sample_limit=None, resume=True, stream=True):
    cmd = [
        sys.executable,
        "-m",
        MODULE,
        "--mode",
        "prod",
        "--run-id",
        run_id,
        "--data-inicio",
        data_inicio,
        "--data-fim",
        data_fim,
    ]
    if resume:
        cmd.append("--resume")
    if sample:
        cmd.append("--sample")
    if sample_limit is not None:
        cmd.extend(["--sample-limit", str(sample_limit)])

    print("Rodando:", " ".join(cmd))
    output_lines = []
    if stream:
        process = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="", flush=True)
            output_lines.append(line.rstrip("\n"))
        returncode = process.wait()
    else:
        result = subprocess.run(cmd, check=False, text=True, capture_output=True)
        returncode = result.returncode
        if result.stdout:
            print(result.stdout)
            output_lines.extend(result.stdout.splitlines())
        if result.stderr:
            print(result.stderr)
            output_lines.extend(result.stderr.splitlines())

    manifest_path = manifest_from_output(output_lines, run_id)
    if returncode != 0:
        print(f"Coletor terminou com returncode={returncode}; a celula continuara para permitir inspecao de logs/autosave.")
    return manifest_path


def manifest_from_output(output_lines, run_id):
    for line in reversed(output_lines):
        candidate = line.strip()
        if candidate.endswith(".json"):
            path = Path(candidate)
            if path.exists():
                return path

    manifest_path = DATA_ROOT / "manifests" / f"{run_id}.json"
    autosave_path = DATA_ROOT / "manifests" / f"{run_id}.autosave.json"
    if manifest_path.exists():
        return manifest_path
    if autosave_path.exists():
        return autosave_path
    return None


def show_manifest(manifest_path):
    if manifest_path is None:
        print("Manifest nao foi informado pelo coletor.")
        return None

    manifest_path = Path(manifest_path)
    if not manifest_path.exists():
        print("Manifest nao encontrado:", manifest_path)
        return None

    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    print("\nManifest:", manifest_path)
    print(json.dumps(manifest, ensure_ascii=False, indent=2)[:6000])
    return manifest


def tail_log_for_run(run_id, n=30):
    log_path = DATA_ROOT / "logs" / f"{run_id}.jsonl"
    print("\nLog:", log_path)
    if not log_path.exists():
        print("Log ainda nao existe.")
        return
    lines = log_path.read_text(encoding="utf-8").splitlines()
    print("\n".join(lines[-n:]))


def inspect_metadata_jsonl(path, max_rows=5):
    path = Path(path)
    print("\nArquivo:", path)
    if not path.exists():
        print("Nao encontrado")
        return

    lines = path.read_text(encoding="utf-8").splitlines()
    print("linhas=", len(lines), "bytes=", path.stat().st_size)
    for line in lines[:max_rows]:
        record = json.loads(line)
        payload = record.get("payload", {})
        summary = {
            "record_type": record.get("record_type"),
            "partition": record.get("partition"),
            "source_id": record.get("source_id"),
        }
        if isinstance(payload, dict):
            summary.update(summarize_payload(payload))
        print(summary)


def assert_no_monthly_corpus_path(source, dataset):
    raw_root = DATA_ROOT / "raw" / source / dataset
    monthly_paths = sorted(raw_root.glob("ano=*/mes=*")) if raw_root.exists() else []
    print("\nParticoes mensais encontradas:", monthly_paths)
    if monthly_paths:
        raise AssertionError("Base metadata-only nao deve criar ano=YYYY/mes=MM")


## Validacao curta

Esta celula roda `coleta.senado.plenario_apartes.collect` em `--mode prod`, com `--sample`, para confirmar que o Drive esta montado e que a base metadata-only grava apenas em `metadata/`.


In [ ]:
manifest_validacao_path = run_collector(
    RUN_ID_VALIDACAO,
    DATA_INICIO_VALIDACAO,
    DATA_FIM_VALIDACAO,
    sample=True,
    sample_limit=SAMPLE_LIMIT_VALIDACAO,
    resume=True,
)
manifest_validacao = show_manifest(manifest_validacao_path)


In [ ]:
metadata_path = DATA_ROOT / "raw" / SOURCE / DATASET / "metadata" / f"{RUN_ID_VALIDACAO}.jsonl"

inspect_metadata_jsonl(metadata_path, max_rows=5)
assert_no_monthly_corpus_path(SOURCE, DATASET)
tail_log_for_run(RUN_ID_VALIDACAO, n=20)


## Coleta completa

Esta celula roda o periodo completo em modo producao, com `run_id` fixo e `--resume`. Se o Colab cair ou a fonte falhar em alguma parte, rode a mesma celula novamente.


In [ ]:
EXECUTAR_COMPLETA = False
RUN_ID_COMPLETA = RUN_ID_PROD

if EXECUTAR_COMPLETA:
    print("Rodando coleta completa com run_id fixo:", RUN_ID_COMPLETA)
    manifest_completa_path = run_collector(
        RUN_ID_COMPLETA,
        DATA_INICIO_PROD,
        DATA_FIM_PROD,
        sample=False,
        sample_limit=None,
        resume=True,
        stream=True,
    )
    manifest_completa = show_manifest(manifest_completa_path)
else:
    print("Altere EXECUTAR_COMPLETA para True quando quiser iniciar a coleta completa.")


## Inspecao e retomada

Use esta celula em outra sessao do Colab para consultar manifesto, autosave e ultimas linhas de log do `run_id` fixo.


In [ ]:
autosave_completa_path = DATA_ROOT / "manifests" / f"{RUN_ID_COMPLETA}.autosave.json"
manifest_completa_path = DATA_ROOT / "manifests" / f"{RUN_ID_COMPLETA}.json"

if manifest_completa_path.exists():
    show_manifest(manifest_completa_path)
elif autosave_completa_path.exists():
    show_manifest(autosave_completa_path)
else:
    print("Ainda nao ha manifest/autosave para", RUN_ID_COMPLETA)

tail_log_for_run(RUN_ID_COMPLETA, n=30)
